# Practical 2: Bayesian Network for Medical Diagnosis
**Problem Statement:**  
Write a program to construct a Bayesian network considering medical data. Use this model to demonstrate the diagnosis of heart patients using standard Heart Disease Data Set. Calculate the accuracy, precision, and recall for your data set. (`heart.csv`)

### Step 1: Import Required Libraries

In [46]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, classification_report

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

print("All libraries imported successfully!")

All libraries imported successfully!


### Step 2: Load and Explore the Dataset

In [47]:
df = pd.read_csv("heart.csv")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (303, 15)

Columns: ['Unnamed: 0', 'Age', 'Sex', 'ChestPain', 'RestBP', 'Chol', 'Fbs', 'RestECG', 'MaxHR', 'ExAng', 'Oldpeak', 'Slope', 'Ca', 'Thal', 'AHD']


,Unnamed: 0,Age,Sex,ChestPain,RestBP,Chol,Fbs,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,AHD
0,1,63,1,typical,145,233,1,2,150,0,2.3,3,0.0,fixed,No
1,2,67,1,asymptomatic,160,286,0,2,108,1,1.5,2,3.0,normal,Yes
2,3,67,1,asymptomatic,120,229,0,2,129,1,2.6,2,2.0,reversable,Yes
3,4,37,1,nonanginal,130,250,0,0,187,0,3.5,3,0.0,normal,No
4,5,41,0,nontypical,130,204,0,2,172,0,1.4,1,0.0,normal,No


### Step 3: Data Preprocessing & Discretization
- Drop index column (`Unnamed: 0`).
- Map target `AHD` (`No` -> 0, `Yes` -> 1).
- Encode categorical `ChestPain`.
- Discretize continuous features (`Age`, `MaxHR`, `Oldpeak`) into 3 quantile bins.

In [48]:
# Drop unneeded index column if present
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Map target AHD
df["AHD"] = df["AHD"].map({"No": 0, "Yes": 1})

# Select diagnostic features
data = df[["Age", "Sex", "ChestPain", "MaxHR", "ExAng", "Oldpeak", "AHD"]].copy()
data = data.dropna()

# Encode categorical ChestPain
data["ChestPain"] = data["ChestPain"].astype("category").cat.codes

# Discretize continuous features
for col in ["Age", "MaxHR", "Oldpeak"]:
    data[col] = pd.qcut(data[col], q=3, labels=False, duplicates="drop")

print("Preprocessed Data:")
data.head()

Preprocessed Data:


,Age,Sex,ChestPain,MaxHR,ExAng,Oldpeak,AHD
0,2,1,3,1,0,2,0
1,2,1,0,0,1,2,1
2,2,1,0,0,1,2,1
3,0,1,1,2,0,2,0
4,0,0,2,2,0,1,0


### Step 4: Split into Training & Testing Sets (80% / 20%)

In [49]:
train, test = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["AHD"]
)

print("Training data:", train.shape)
print("Testing data :", test.shape)

Training data: (242, 7)
Testing data : (61, 7)


### Step 5: Construct the Bayesian Network (DAG Structure)

In [50]:
model = DiscreteBayesianNetwork([
    ("AHD", "Age"),
    ("AHD", "Sex"),
    ("AHD", "ChestPain"),
    ("AHD", "MaxHR"),
    ("AHD", "ExAng"),
    ("AHD", "Oldpeak")
])

print("Nodes in Bayesian Network:", model.nodes())
print("Edges in Bayesian Network:", model.edges())

Nodes in Bayesian Network: ['AHD', 'Age', 'Sex', 'ChestPain', 'MaxHR', 'ExAng', 'Oldpeak']
Edges in Bayesian Network: [('AHD', 'Age'), ('AHD', 'Sex'), ('AHD', 'ChestPain'), ('AHD', 'MaxHR'), ('AHD', 'ExAng'), ('AHD', 'Oldpeak')]


### Step 6: Parameter Estimation (Learning CPDs)

In [51]:
estimator = BayesianEstimator(model, train)
cpds = estimator.get_parameters(prior_type="BDeu", equivalent_sample_size=10)
model.add_cpds(*cpds)

print("Model valid:", model.check_model())

Model valid: True


C:\Users\mkasw\AppData\Local\Temp\ipykernel_8628\2404326511.py:1: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(model, train)


### Step 7: Demonstrate Heart Patient Diagnosis (Probabilistic Inference)

In [52]:
infer = VariableElimination(model)

# Case 1: High-Risk Patient
high_risk = {
    "Age": 2,        # High age quantile
    "Sex": 1,        # Male
    "ChestPain": 0,  # Asymptomatic chest pain
    "MaxHR": 0,      # Low max heart rate
    "ExAng": 1,      # Exercise angina present
    "Oldpeak": 2     # High ST depression
}
q1 = infer.query(variables=["AHD"], evidence=high_risk, show_progress=False)
print("--- Demonstration Case 1: High-Risk Patient ---")
print("Evidence:", high_risk)
print(f"P(No Disease [0]): {q1.values[0]:.4f} ({q1.values[0]*100:.2f}%)")
print(f"P(Disease    [1]): {q1.values[1]:.4f} ({q1.values[1]*100:.2f}%)")
print("Diagnosis Result :", "POSITIVE FOR HEART DISEASE" if q1.values.argmax() == 1 else "NEGATIVE")

# Case 2: Low-Risk Patient
low_risk = {
    "Age": 0,        # Low age quantile
    "Sex": 0,        # Female
    "ChestPain": 1,  # Non-anginal pain
    "MaxHR": 2,      # High max heart rate
    "ExAng": 0,      # No exercise angina
    "Oldpeak": 0     # Low ST depression
}
q2 = infer.query(variables=["AHD"], evidence=low_risk, show_progress=False)
print("\n--- Demonstration Case 2: Low-Risk Patient ---")
print("Evidence:", low_risk)
print(f"P(No Disease [0]): {q2.values[0]:.4f} ({q2.values[0]*100:.2f}%)")
print(f"P(Disease    [1]): {q2.values[1]:.4f} ({q2.values[1]*100:.2f}%)")
print("Diagnosis Result :", "POSITIVE FOR HEART DISEASE" if q2.values.argmax() == 1 else "NEGATIVE")

--- Demonstration Case 1: High-Risk Patient ---
Evidence: {'Age': 2, 'Sex': 1, 'ChestPain': 0, 'MaxHR': 0, 'ExAng': 1, 'Oldpeak': 2}
P(No Disease [0]): 0.0073 (0.73%)
P(Disease    [1]): 0.9927 (99.27%)
Diagnosis Result : POSITIVE FOR HEART DISEASE

--- Demonstration Case 2: Low-Risk Patient ---
Evidence: {'Age': 0, 'Sex': 0, 'ChestPain': 1, 'MaxHR': 2, 'ExAng': 0, 'Oldpeak': 0}
P(No Disease [0]): 0.9933 (99.33%)
P(Disease    [1]): 0.0067 (0.67%)
Diagnosis Result : NEGATIVE


### Step 8: Predict on Test Data

In [53]:
evidence_cols = ["Age", "Sex", "ChestPain", "MaxHR", "ExAng", "Oldpeak"]
predictions = []

for _, row in test.iterrows():
    evidence = {col: row[col] for col in evidence_cols}
    q = infer.query(variables=["AHD"], evidence=evidence, show_progress=False)
    predictions.append(int(q.values.argmax()))

print("Actual values (first 10)   :", test["AHD"].values[:10])
print("Predicted values (first 10):", predictions[:10])

Actual values (first 10)   : [0 0 0 0 0 0 0 0 1 0]
Predicted values (first 10): [0, 1, 0, 0, 0, 0, 0, 1, 1, 1]


### Step 9: Evaluate Performance (Accuracy, Precision, Recall, Confusion Matrix)

In [54]:
y_test = test["AHD"].astype(int).values

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)

print("=========================================")
print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall   : {recall * 100:.2f}%")
print("=========================================")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=["No Heart Disease (0)", "Heart Disease (1)"]))

Accuracy : 86.89%
Precision: 83.33%
Recall   : 89.29%

Confusion Matrix:
[[28  5]
 [ 3 25]]

Classification Report:
                      precision    recall  f1-score   support

No Heart Disease (0)       0.90      0.85      0.88        33
   Heart Disease (1)       0.83      0.89      0.86        28

            accuracy                           0.87        61
           macro avg       0.87      0.87      0.87        61
        weighted avg       0.87      0.87      0.87        61

